In [0]:
#total revenue collected
fact_sales = spark.read.table("de_mini_project.gold.fact_sales")
revenue_df = fact_sales.filter(fact_sales["return_id"].isNull()).withColumn("revenue", fact_sales["qty_sold"] * fact_sales["unit_price"])
display(revenue_df)
revenue = revenue_df.agg({"revenue": "sum"}).collect()[0][0]
print("total revenue",revenue)

In [0]:
#profit per product
fact_inventory = spark.read.table("de_mini_project.gold.fact_inventory")
profit_df = fact_inventory.withColumn("profit", fact_inventory["marked_price"] - fact_inventory["base_cost"]).drop("stock_on_hand").select("sku_id","base_cost","marked_price","profit").distinct().collect()
display(profit_df)

In [0]:
#most sold category
dim_inventory = spark.read.table("de_mini_project.gold.dim_inventory")
category_df = revenue_df.join(dim_inventory,["sku_id"],"left")
category_df = category_df.select("transaction_id","category","revenue").distinct()
top_category = category_df.groupBy("category").agg({"revenue": "sum"}).sort("sum(revenue)",ascending=False).limit(1)
display(top_category)

In [0]:
#avg number of items bought per transactions
avg_items = fact_sales.agg({"qty_sold":"avg"}).withColumnRenamed("avg(qty_sold)","average")
display(avg_items)

In [0]:
#return rate
return_count = fact_sales.agg({"return_id":"count"})
total_sales_count = fact_sales.count()
return_rate = (return_count.collect()[0][0]/total_sales_count)*100
print("return rate",return_rate,"%")

In [0]:
#out of stock count
out_of_stock_count = fact_inventory.filter(fact_inventory["stock_on_hand"]==0).count()
print("out of stock count",out_of_stock_count)


In [0]:
#store performance
dim_stores = spark.read.table("de_mini_project.gold.dim_stores")
stores_df = fact_sales.withColumn("revenue", fact_sales["qty_sold"] * fact_sales["unit_price"])
stores_df = stores_df.groupBy("store_id").agg({"revenue": "sum"})
stores_df = stores_df.join(dim_stores,["store_id"],"left").sort("sum(revenue)",ascending=False)
display(stores_df)

In [0]:
#discount impact
from pyspark.sql.functions import round

discount_impact_df = fact_sales.join(fact_inventory,["sku_id"],"left").drop("return_date","return_id","stock_on_hand","customer_id","store_id")
discount_impact_df = discount_impact_df.withColumn("loss", round(discount_impact_df["marked_price"] - discount_impact_df["unit_price"], 2)*discount_impact_df["qty_sold"])
display(discount_impact_df)

In [0]:
#repeat customer count
repeat_customer_df = fact_sales.groupBy("customer_id").agg({"transaction_id": "count"}).withColumnRenamed("count(transaction_id)","order_count").filter(fact_sales["customer_id"]!= 'Unkown Customer')
display(repeat_customer_df)

In [0]:
#slow moving inventory
from datetime import date, timedelta
today = date.today()
days_ago = today - timedelta(days=60)
slow_moving_inv_df = fact_sales.drop("customer_id","store_id","unit_price","qty_sold","transaction_id","return_date","return_id").filter(fact_sales['date']>days_ago_30)
slow_moving_inv_df = fact_inventory.join(slow_moving_inv_df,["sku_id"],"left").drop("stock_on_hand","store_id")

slow_moving_inv_df = slow_moving_inv_df.filter(slow_moving_inv_df['date'].isNull()).join(dim_inventory,["sku_id"],"left").drop("date")
display(slow_moving_inv_df)